# Phase 3 — Seaborn for Statistical Visualization

Seaborn is built on matplotlib and makes statistical plots much easier. It integrates directly with pandas DataFrames and comes with built-in datasets.

**Key idea:** Seaborn maps **data columns** directly to **visual properties** (x, y, hue, size, col, row).

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# Load built-in datasets for practice
tips = sns.load_dataset("tips")  # restaurant tips
iris = sns.load_dataset("iris")  # flower measurements
titanic = sns.load_dataset("titanic")  # passenger survival

print("tips shape:", tips.shape)
print("iris shape:", iris.shape)
print("titanic shape:", titanic.shape)

---
## 1. Distribution Plots

Use these to visualize the **shape, spread, and central tendency** of data — core descriptive statistics concepts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# histplot — histogram with optional KDE
sns.histplot(
    data=tips, x="total_bill", bins=20, kde=True, ax=axes[0], color="steelblue"
)
axes[0].set_title("histplot — Total Bill Distribution")

# kdeplot — smooth density estimate
sns.kdeplot(data=tips, x="total_bill", hue="time", fill=True, alpha=0.4, ax=axes[1])
axes[1].set_title("kdeplot — Bill by Meal Time")

# ecdfplot — empirical cumulative distribution (CDF from data)
sns.ecdfplot(data=tips, x="total_bill", hue="sex", ax=axes[2])
axes[2].set_title("ecdfplot — Empirical CDF by Gender")

plt.tight_layout()
plt.show()

---
## 2. Categorical Plots

Use these to compare distributions across groups.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# boxplot — shows Q1, Q2, Q3, IQR, whiskers, outliers
sns.boxplot(data=tips, x="day", y="total_bill", hue="sex", ax=axes[0, 0])
axes[0, 0].set_title("boxplot — Bill by Day & Gender")

# violinplot — like boxplot but shows the distribution shape
sns.violinplot(
    data=tips, x="day", y="tip", hue="sex", split=True, ax=axes[0, 1], inner="quartile"
)
axes[0, 1].set_title("violinplot — Tips by Day & Gender")

# stripplot — all individual data points
sns.stripplot(
    data=tips,
    x="day",
    y="total_bill",
    hue="sex",
    dodge=True,
    alpha=0.5,
    ax=axes[0, 2],
    jitter=True,
)
axes[0, 2].set_title("stripplot — All Data Points")

# barplot — mean + confidence interval
sns.barplot(
    data=tips,
    x="day",
    y="total_bill",
    hue="sex",
    ax=axes[1, 0],
    errorbar="ci",
    capsize=0.1,
)
axes[1, 0].set_title("barplot — Mean ± 95% CI")

# pointplot — same as barplot but as points with lines
sns.pointplot(
    data=tips, x="day", y="tip", hue="sex", dodge=0.3, ax=axes[1, 1], capsize=0.1
)
axes[1, 1].set_title("pointplot — Mean Tip by Day")

# countplot — counts of categorical data
sns.countplot(data=titanic, x="class", hue="survived", ax=axes[1, 2])
axes[1, 2].set_title("countplot — Survival by Class")

plt.suptitle("Seaborn Categorical Plots", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Relational Plots

Use these to show **relationships between variables** — directly tied to regression and correlation analysis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# scatterplot
sns.scatterplot(
    data=tips, x="total_bill", y="tip", hue="time", size="size", ax=axes[0], alpha=0.7
)
axes[0].set_title("scatterplot — Bill vs Tip")

# regplot — scatter + regression line + confidence band
sns.regplot(
    data=tips,
    x="total_bill",
    y="tip",
    ax=axes[1],
    scatter_kws={"alpha": 0.5, "s": 30},
    line_kws={"color": "red", "lw": 2},
)
axes[1].set_title("regplot — Regression Line + CI")

# residplot — plot of regression residuals (diagnostic tool)
sns.residplot(
    data=tips, x="total_bill", y="tip", ax=axes[2], scatter_kws={"alpha": 0.5, "s": 30}
)
axes[2].set_title("residplot — Regression Residuals")

plt.tight_layout()
plt.show()

---
## 4. Heatmaps — Correlation Matrices

Essential for EDA: which features are correlated? This directly connects to what you learned about Pearson r.

In [ ]:
# Correlation heatmap
corr_matrix = iris.drop("species", axis=1).corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full correlation matrix
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=axes[0],
)
axes[0].set_title("Correlation Matrix — Iris Features")

# Upper triangle only (avoid redundancy)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=axes[1],
)
axes[1].set_title("Lower Triangle Only")

plt.tight_layout()
plt.show()

---
## 5. pairplot — The One-Line EDA Tool

`pairplot` creates a full grid of scatter plots and histograms for all variable pairs. This is the first plot you should make on any new dataset.

In [ ]:
g = sns.pairplot(
    iris,
    hue="species",  # color by category
    diag_kind="kde",  # diagonal: KDE density
    plot_kws={"alpha": 0.6, "s": 30},
)
g.figure.suptitle("Pairplot — Iris Dataset", y=1.01, fontsize=14)
plt.show()

---
## 6. FacetGrid — Conditional Plots

In [ ]:
# Facet by a category — one subplot per category
g = sns.FacetGrid(tips, col="time", row="sex", height=3.5, aspect=1.2)
g.map(sns.histplot, "total_bill", bins=15, kde=True, color="steelblue")
g.set_titles("{col_name} | {row_name}")
g.set_axis_labels("Total Bill ($)", "Count")
g.figure.suptitle("Bill Distribution by Meal Time and Gender", y=1.02)
plt.show()

---
## Summary — Seaborn Cheat Sheet

| Plot Type | Function | When to Use |
|-----------|----------|-------------|
| Distribution | `histplot`, `kdeplot`, `ecdfplot` | Show data distribution, check normality |
| Box & Violin | `boxplot`, `violinplot` | Compare distributions across groups |
| Points | `scatterplot`, `stripplot` | Show all data points, find outliers |
| Summary | `barplot`, `pointplot` | Show means with confidence intervals |
| Regression | `regplot`, `residplot` | Visualize linear relationships |
| Heatmap | `heatmap` | Correlation matrices, 2D data |
| All pairs | `pairplot` | First EDA step on any dataset |
| Conditional | `FacetGrid` | One plot per category |

**Seaborn's `hue` parameter** lets you split any plot by a categorical variable — no manual looping needed.